In [1]:
import json
import random
from pathlib import Path

N_PARAGRAPHS = 5
REGISTER     = "peer-reviewed journal article"

BASE_DIR    = Path(".")
SPECIES_DIR = BASE_DIR / "species_data"
LEXICON_DIR = BASE_DIR / "lexical_data"
PROMPT_FILE = BASE_DIR / "prompt_claude.md"
OUTPUT_DIR  = BASE_DIR / "prompts_to_send"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def get_taxon(taxonomy: dict, key: str, default: str = "") -> str:
    value = taxonomy.get(key, "")
    if not value:
        alt_key = key + "s" if not key.endswith("s") else key[:-1]
        value = taxonomy.get(alt_key, default)
    return value


def load_prompt_template(path: Path) -> str:
    text = path.read_text()
    lines = text.strip().splitlines()
    if lines[0].startswith("```"):
        lines = lines[1:]
    if lines[-1].startswith("```"):
        lines = lines[:-1]
    return "\n".join(lines)


def load_lexicon(cls: str, deb_model: str) -> list:
    lex_file = LEXICON_DIR / f"{cls}_{deb_model}.json"
    raw = json.loads(lex_file.read_text())
    top_key = next(iter(raw))
    return raw[top_key]["stades"]


def kelvin_to_celsius(temp_raw: str) -> str | None:
    """Convert C2K(x) Kelvin notation to Celsius string, e.g. 'C2K(5)' → '5'.
    Returns None if no valid temperature found."""
    if not temp_raw or str(temp_raw) in ("", "nan", "None", "null"):
        return None
    import re
    # C2K(x) means x°C converted to Kelvin — extract the Celsius value directly
    m = re.match(r"C2K\(([^)]+)\)", str(temp_raw))
    if m:
        return m.group(1)
    # If it's a plain Kelvin number, convert: °C = K - 273.15
    try:
        k = float(temp_raw)
        c = k - 273.15
        return str(round(c, 2)) if c != int(c) else str(int(c))
    except ValueError:
        return None


def format_entity_annotations(measurements: list) -> str:
    lines = []
    for m in measurements:
        entity  = m.get("entity", "?")
        value   = m.get("entity.value", "?")
        units   = m.get("units", "")
        label   = m.get("label", "")
        comment = m.get("comment", "")
        celsius = kelvin_to_celsius(m.get("temp", ""))

        line = f"- {entity} = {value} {units}"
        if celsius is not None:
            line += f"  @ {celsius}°C"
        line += f"  [{label}]"
        if comment and str(comment) not in ("", "nan", "None"):
            line += f"  // {comment}"
        if celsius is not None:
            line += f"  → temperature must be annotated as [T = {celsius} °C]"
        lines.append(line)
    return "\n".join(lines) if lines else "(no measurements available)"


def format_full_lexicon(stades: list) -> str:
    sections = []
    for stade in stades:
        name  = stade.get("nom", stade.get("label", "Stage"))
        terms = stade.get("termes", [])
        block  = f"### Stage: {name}\n"
        block += "\n".join(f"  - {t}" for t in terms)
        sections.append(block)
    return "\n\n".join(sections) if sections else "(no lexicon terms available)"


def pick_transition(stades: list) -> tuple:
    if len(stades) >= 2:
        idx = random.randint(0, len(stades) - 2)
        a = stades[idx].get("nom", stades[idx].get("label", "earlier stage"))
        b = stades[idx + 1].get("nom", stades[idx + 1].get("label", "later stage"))
        return f"{a} to {b}", a, b
    return "life stage transition", "juvenile", "adult"


def build_prompt(species: dict, stades: list, template: str, n_paragraphs: int, register: str) -> str:
    taxonomy     = species["taxonomy"]
    measurements = species["data"]
    event_name, from_stage, to_stage = pick_transition(stades)
    species_name  = get_taxon(taxonomy, "specie_en", get_taxon(taxonomy, "specie", "Unknown")).replace("_", " ")
    species_latin = get_taxon(taxonomy, "specie", "Unknown").replace("_", " ")
    tax_class     = taxonomy.get("class", "")
    filled = template.replace("{N_PARAGRAPHS}",       str(n_paragraphs))
    filled = filled.replace("{SPECIES_NAME}",         species_name)
    filled = filled.replace("{SPECIES_LATIN}",        species_latin)
    filled = filled.replace("{TAXONOMIC_CLASS}",      tax_class)
    filled = filled.replace("{ENTITY_ANNOTATIONS}",   format_entity_annotations(measurements))
    filled = filled.replace("{LIFECYCLE_EVENT_NAME}", event_name)
    filled = filled.replace("{FROM_STAGE}",           from_stage)
    filled = filled.replace("{TO_STAGE}",             to_stage)
    filled = filled.replace("{LEXICON_TERMS}",        format_full_lexicon(stades))
    filled = filled.replace("{REGISTER}",             register)
    return filled


In [3]:
template      = load_prompt_template(PROMPT_FILE)
valid_species = []
skipped       = []

for f in sorted(SPECIES_DIR.glob("*.json")):
    try:
        sp        = json.loads(f.read_text())
        cls       = sp["taxonomy"]["class"]
        deb_model = sp["taxonomy"]["deb_model"]
        lex_file  = LEXICON_DIR / f"{cls}_{deb_model}.json"
        if lex_file.exists():
            valid_species.append(sp)
        else:
            skipped.append(f"{f.name}  (no lexicon for {cls}_{deb_model})")
    except Exception as e:
        skipped.append(f"{f.name}  ({e})")

print(f"Valid species : {len(valid_species)}")
print(f"Skipped       : {len(skipped)}")
if skipped:
    for s in skipped[:10]:
        print(f"  {s}")

Valid species : 7281
Skipped       : 0


In [4]:
already_saved   = {f.stem.split("_", 1)[1].rsplit("_", 1)[0] for f in OUTPUT_DIR.glob("*.json")}
prompts_saved   = 0
prompts_skipped = 0

for i, sp in enumerate(valid_species, 1):
    taxonomy  = sp["taxonomy"]
    cls       = taxonomy["class"]
    deb_model = taxonomy["deb_model"]
    specie    = get_taxon(taxonomy, "specie")
    specie_en = get_taxon(taxonomy, "specie_en", specie)

    if specie in already_saved:
        prompts_skipped += 1
        continue

    stades   = load_lexicon(cls, deb_model)
    prompt   = build_prompt(sp, stades, template, N_PARAGRAPHS, REGISTER)
    filename = f"{i:04d}_{specie.replace(' ', '_')}_{deb_model}.json"

    payload = {
        "id":        i,
        "specie":    specie,
        "specie_en": specie_en,
        "deb_model": deb_model,
        "prompt":    prompt,
        "result":    None,
    }
    (OUTPUT_DIR / filename).write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    prompts_saved += 1

print(f"Saved   : {prompts_saved}")
print(f"Skipped : {prompts_skipped} (already existed)")
print(f"Total in OUTPUT_DIR : {len(list(OUTPUT_DIR.glob('*.json')))}")


Saved   : 7281
Skipped : 0 (already existed)
Total in OUTPUT_DIR : 7281


In [5]:
files   = sorted(OUTPUT_DIR.glob("*.json"))
done    = [f for f in files if json.loads(f.read_text()).get("result") is not None]
pending = [f for f in files if json.loads(f.read_text()).get("result") is None]

print(f"Total prompt files : {len(files)}")
print(f"  result filled    : {len(done)}")
print(f"  result = None    : {len(pending)}  (ready to send to API)")

Total prompt files : 7281
  result filled    : 0
  result = None    : 7281  (ready to send to API)
